# 🌾 Finetune YOLOv26x – Farm Field Boundary Segmentation

| Item | Detail |
|------|--------|
| Task | Instance segmentation |
| Base model | `yolo26x-seg` (extra-large) |
| Classes | 1 — `land-Sx1C` (farm/cropland boundaries) |
| Dataset | 62 images (43 train / 13 valid / 6 test), 640×640 |
| Framework | [Ultralytics](https://docs.ultralytics.com/) |
| Tracking | **MLflow** + **TensorBoard** |

**Idempotent**: Every cell checks if its outputs already exist and skips if so.  
Safe to re-run the entire notebook without re-downloading or re-training.

## 0️⃣ Environment Setup

In [ ]:
import os
os.environ['http_proxy']  = 'http://10.68.69.53:80/'
os.environ['https_proxy'] = 'http://10.68.69.53:80/'

import numpy as np
import torch
import ultralytics
from pathlib import Path

NOTEBOOK_DIR = Path('.').resolve()

print(f'NumPy        : {np.__version__}')
print(f'PyTorch      : {torch.__version__}')
print(f'Ultralytics  : {ultralytics.__version__}')
print(f'Notebook dir : {NOTEBOOK_DIR}')

t = torch.from_numpy(np.array([1.0, 2.0, 3.0]))
print(f'torch.from_numpy test: {t}  ✅')

if torch.cuda.is_available():
    print(f'CUDA         : {torch.cuda.get_device_name(0)}')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    print(f'MPS (Apple)  : available ✅')
else:
    print(f'Device       : CPU only')

## 1️⃣ Setup MLflow & TensorBoard Tracking

In [ ]:
import mlflow
from ultralytics import settings

MLFLOW_DIR  = NOTEBOOK_DIR / 'mlruns'
EXPERIMENT  = 'Farm-Field-Segmentation'

os.environ['MLFLOW_TRACKING_URI'] = f'file://{MLFLOW_DIR}'

settings.update({'mlflow': True, 'tensorboard': True})

mlflow.set_tracking_uri(f'file://{MLFLOW_DIR}')
mlflow.set_experiment(EXPERIMENT)

print(f'✅ MLflow  : file://{MLFLOW_DIR}')
print(f'   Experiment : {EXPERIMENT}')
print(f'\n📊 View dashboards:')
print(f'   mlflow ui --backend-store-uri file://{MLFLOW_DIR} --host 0.0.0.0 --port 5000')
print(f'   tensorboard --logdir {NOTEBOOK_DIR / "farm_fields_seg"} --host 0.0.0.0 --port 6006')

## 2️⃣ Load Model

In [ ]:
import zipfile
from ultralytics import YOLO

MODEL_PATH   = '/mnt/raid1-backup/houcine/models/yolo26x-seg.pt'
DOWNLOAD_URL = 'https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26x-seg.pt'

IMAGE_SIZE   = 640
PROJECT      = str(NOTEBOOK_DIR / 'farm_fields_seg')
RUN_NAME     = 'yolo26x_run2'

def is_valid_pt(path):
    if not os.path.exists(path) or os.path.getsize(path) < 1_000_000:
        return False
    try:
        with zipfile.ZipFile(path, 'r') as z:
            return len(z.namelist()) > 0
    except Exception:
        return False

if not is_valid_pt(MODEL_PATH):
    import subprocess
    print(f'⚠️  Downloading model…')
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    subprocess.run(['curl', '-L', '--retry', '3', '--connect-timeout', '60',
                    '-o', MODEL_PATH, DOWNLOAD_URL], capture_output=True, text=True)

if is_valid_pt(MODEL_PATH):
    print(f'✅ Model ready: {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)')
    model = YOLO(MODEL_PATH)
    HAS_PRETRAINED = True
else:
    print(f'❌ Falling back to YAML (train from scratch).')
    model = YOLO('yolo26x-seg.yaml')
    HAS_PRETRAINED = False

print(f'Pretrained: {HAS_PRETRAINED}')

## 3️⃣ Prepare `data.yaml`

**Always regenerated** — paths are machine-specific and must match the current system.

In [ ]:
import yaml

DATASET_DIR = NOTEBOOK_DIR / 'Farm Fields.v1i.yolov8'
TRAIN_YAML  = DATASET_DIR / 'data_train.yaml'

# Always regenerate — paths are machine-specific!
with open(DATASET_DIR / 'data.yaml') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path']  = str(DATASET_DIR)
data_cfg['train'] = 'train/images'
data_cfg['val']   = 'valid/images'
data_cfg['test']  = 'test/images'

with open(TRAIN_YAML, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f'✅ data_train.yaml written with paths for THIS machine:')
print(f'   path: {data_cfg["path"]}')
print(open(TRAIN_YAML).read())

## 4️⃣ Sanity Check — Visualise Samples

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

%matplotlib inline

img_dir = DATASET_DIR / 'train' / 'images'
lbl_dir = DATASET_DIR / 'train' / 'labels'

all_imgs = sorted(img_dir.glob('*.jpg'))
samples = random.sample(all_imgs, min(4, len(all_imgs)))

fig, axes = plt.subplots(1, len(samples), figsize=(20, 5))
for ax, img_path in zip(axes, samples):
    img = np.array(Image.open(img_path))
    ax.imshow(img)

    lbl_path = lbl_dir / f'{img_path.stem}.txt'
    if lbl_path.exists():
        h, w = img.shape[:2]
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                coords = list(map(float, parts[1:]))
                xs = np.array(coords[0::2]) * w
                ys = np.array(coords[1::2]) * h
                ax.fill(xs, ys, alpha=0.3, fc='lime', ec='lime', lw=1.2)

    ax.set_title(img_path.name[:20] + '…', fontsize=8)
    ax.axis('off')

fig.suptitle('Training samples with segmentation masks', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5️⃣ Save Pretrained (COCO) Predictions BEFORE Finetuning

In [ ]:
test_images = sorted((DATASET_DIR / 'test' / 'images').glob('*.jpg'))

pretrained_plots = {}
if HAS_PRETRAINED:
    pretrained_preds = model.predict(
        source=[str(p) for p in test_images], imgsz=IMAGE_SIZE, conf=0.25, save=False, verbose=False)
    for result in pretrained_preds:
        pretrained_plots[Path(result.path).name] = result.plot(boxes=False)[:, :, ::-1]
    print(f'✅ Saved pretrained (COCO) predictions for {len(pretrained_plots)} test images')
else:
    print('⏭️  Skipping — no pretrained weights')

## 6️⃣ Finetune YOLOv26x-seg (with MLflow Tracking)

Skips if `best.pt` already exists.

In [ ]:
run_dir  = Path(PROJECT) / RUN_NAME
best_wt  = run_dir / 'weights' / 'best.pt'

if best_wt.exists():
    print(f'⏭️  Training already complete — {best_wt} exists ({best_wt.stat().st_size/1e6:.1f} MB)')
    print(f'   Delete {run_dir} to retrain.')
else:
    with mlflow.start_run(run_name=RUN_NAME) as run:
        mlflow.set_tag('model_arch', 'yolo26x-seg')
        mlflow.set_tag('dataset', 'Farm Fields v1i')
        mlflow.set_tag('task', 'instance_segmentation')
        mlflow.set_tag('pretrained', str(HAS_PRETRAINED))

        train_params = dict(
            data         = str(TRAIN_YAML),
            epochs       = 300,
            imgsz        = IMAGE_SIZE,
            batch        = 4,
            patience     = 50,
            project      = PROJECT,
            name         = RUN_NAME,
            exist_ok     = True,
            plots        = True,
            save         = True,
            save_period  = 50,
            val          = True,
            pretrained   = HAS_PRETRAINED,
            optimizer    = 'AdamW',
            lr0          = 0.001,
            lrf          = 0.01,
            cos_lr       = True,
            warmup_epochs = 5.0,
            warmup_momentum = 0.8,
            weight_decay = 0.0005,
            mosaic       = 1.0,
            close_mosaic = 20,
            mixup        = 0.15,
            copy_paste   = 0.5,
            flipud       = 0.5,
            fliplr       = 0.5,
            degrees      = 15.0,
            translate    = 0.2,
            scale        = 0.5,
            shear        = 2.0,
            perspective  = 0.001,
            hsv_h        = 0.02,
            hsv_s        = 0.7,
            hsv_v        = 0.4,
            erasing      = 0.3,
            label_smoothing = 0.05,
            nbs          = 64,
            iou          = 0.6,
            workers      = 0,
        )

        mlflow.log_params({k: str(v) for k, v in train_params.items()})
        results = model.train(**train_params)

        if hasattr(results, 'results_dict'):
            for k, v in results.results_dict.items():
                try:
                    mlflow.log_metric(k.replace('/', '_').replace('(', '').replace(')', ''), float(v))
                except (TypeError, ValueError):
                    pass

        if best_wt.exists():
            mlflow.log_artifact(str(best_wt), artifact_path='model')
        for pn in ['results.png', 'confusion_matrix.png']:
            pp = run_dir / pn
            if pp.exists():
                mlflow.log_artifact(str(pp), artifact_path='plots')

        print(f'\n✅ Training complete!  MLflow run: {run.info.run_id}')

## 7️⃣ Training Curves

In [ ]:
for pf in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
           'val_batch0_labels.jpg', 'val_batch0_pred.jpg']:
    p = run_dir / pf
    if p.exists():
        fig, ax = plt.subplots(figsize=(14, 8))
        ax.imshow(np.array(Image.open(p)))
        ax.set_title(pf, fontsize=12, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f'⚠️  {pf} not found')

## 8️⃣ Evaluate on Val & Test

In [ ]:
best_model = YOLO(str(best_wt))

for split_name in ['val', 'test']:
    m = best_model.val(data=str(TRAIN_YAML), split=split_name)
    print(f'\n{"="*50}')
    print(f'  {split_name.upper()} Results (Segmentation)')
    print(f'{"="*50}')
    print(f'  mAP50       : {m.seg.map50:.4f}')
    print(f'  mAP50-95    : {m.seg.map:.4f}')
    print(f'  Precision   : {m.seg.mp:.4f}')
    print(f'  Recall      : {m.seg.mr:.4f}')
    print(f'{"="*50}')

## 9️⃣ MLflow Experiment Dashboard

In [ ]:
import pandas as pd

experiment = mlflow.get_experiment_by_name(EXPERIMENT)
if experiment is not None:
    runs_df = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=['start_time DESC']
    )
    if not runs_df.empty:
        key_cols = ['run_id', 'status', 'start_time']
        metric_cols = [c for c in runs_df.columns if c.startswith('metrics.')]
        param_cols  = [c for c in runs_df.columns if c.startswith('params.') and
                       any(k in c for k in ['epochs', 'lr0', 'batch', 'optimizer', 'imgsz'])]
        tag_cols    = [c for c in runs_df.columns if c.startswith('tags.') and 'mlflow' not in c]
        display_cols = [c for c in key_cols + tag_cols + param_cols + metric_cols if c in runs_df.columns]
        display_df = runs_df[display_cols].copy()
        display_df.columns = [c.replace('metrics.', '📈 ').replace('params.', '⚙️ ').replace('tags.', '🏷️ ') for c in display_df.columns]
        print(f'📊 {EXPERIMENT} — {len(display_df)} run(s)\n')
        display(display_df)
    else:
        print('No runs yet.')
else:
    print(f'Experiment "{EXPERIMENT}" not found yet.')

---

# 🔬 Comparisons

---

## 🔟 Ground Truth vs YOLOv26x Predictions

In [ ]:
def draw_gt(img_path, lbl_dir):
    img = np.array(Image.open(img_path))
    fig_tmp, ax_tmp = plt.subplots(figsize=(6, 6))
    ax_tmp.imshow(img)
    lbl_path = lbl_dir / f'{Path(img_path).stem}.txt'
    n_polys = 0
    if lbl_path.exists():
        h, w = img.shape[:2]
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                coords = list(map(float, parts[1:]))
                xs = np.array(coords[0::2]) * w
                ys = np.array(coords[1::2]) * h
                ax_tmp.fill(xs, ys, alpha=0.35, fc='lime', ec='lime', lw=1.5)
                n_polys += 1
    ax_tmp.axis('off')
    fig_tmp.tight_layout(pad=0)
    fig_tmp.canvas.draw()
    buf = np.asarray(fig_tmp.canvas.buffer_rgba())
    rendered = buf[:, :, :3].copy()
    plt.close(fig_tmp)
    return rendered, n_polys

In [ ]:
INFER_CONF = 0.15

finetuned_preds = best_model.predict(
    source=[str(p) for p in test_images], imgsz=IMAGE_SIZE, conf=INFER_CONF, save=False, verbose=False)

n = len(test_images)
fig, axes = plt.subplots(n, 2, figsize=(16, 7 * n))
if n == 1:
    axes = axes[np.newaxis, :]

for i, (img_path, result) in enumerate(zip(test_images, finetuned_preds)):
    gt_img, gt_count = draw_gt(str(img_path), DATASET_DIR / 'test' / 'labels')
    axes[i, 0].imshow(gt_img)
    axes[i, 0].set_title(f'Ground Truth — {gt_count} polygons', fontsize=11, fontweight='bold', color='lime')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(result.plot(boxes=False)[:, :, ::-1])
    n_masks = len(result.masks) if result.masks is not None else 0
    axes[i, 1].set_title(f'YOLOv26x — {n_masks} fields (conf≥{INFER_CONF})', fontsize=11, fontweight='bold', color='cyan')
    axes[i, 1].axis('off')

fig.suptitle('A) Ground Truth vs YOLOv26x Delineation', fontsize=15, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

## 1️⃣1️⃣ Before vs After Finetuning

In [ ]:
if not pretrained_plots:
    print('⏭️  Skipping — no pretrained predictions saved')
else:
    n = len(test_images)
    fig, axes = plt.subplots(n, 2, figsize=(16, 7 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    for i, (img_path, ft_result) in enumerate(zip(test_images, finetuned_preds)):
        name = img_path.name
        if name in pretrained_plots:
            axes[i, 0].imshow(pretrained_plots[name])
        else:
            axes[i, 0].imshow(np.array(Image.open(img_path)))
        axes[i, 0].set_title('BEFORE — pretrained COCO', fontsize=11, fontweight='bold', color='orange')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(ft_result.plot(boxes=False)[:, :, ::-1])
        n_masks = len(ft_result.masks) if ft_result.masks is not None else 0
        axes[i, 1].set_title(f'AFTER — finetuned ({n_masks} fields)', fontsize=11, fontweight='bold', color='cyan')
        axes[i, 1].axis('off')

    fig.suptitle('B) Before vs After Finetuning', fontsize=15, fontweight='bold', y=1.005)
    plt.tight_layout()
    plt.show()

## 1️⃣2️⃣ Export Model

In [ ]:
onnx_path = run_dir / 'weights' / 'best.onnx'

if onnx_path.exists():
    print(f'⏭️  ONNX already exported: {onnx_path} ({onnx_path.stat().st_size/1e6:.1f} MB)')
else:
    best_model.export(format='onnx', imgsz=IMAGE_SIZE, simplify=True)
    if onnx_path.exists():
        try:
            with mlflow.start_run(run_name=f'{RUN_NAME}_export'):
                mlflow.log_artifact(str(onnx_path), artifact_path='model')
        except Exception:
            pass
    print(f'✅ Exported: {onnx_path}')

print(f'\n📁 Outputs: {run_dir.resolve()}')

## 1️⃣3️⃣ Inference on Custom Image

In [ ]:
CUSTOM_IMAGE = Path('/mnt/raid1-backup/houcine/field_delineation_yolo/75_jpg.rf.f423d98d6d6ce09dfedef22e658ccec8.jpg')

assert CUSTOM_IMAGE.exists(), f'Image not found: {CUSTOM_IMAGE}'

result = best_model.predict(str(CUSTOM_IMAGE), imgsz=IMAGE_SIZE, conf=INFER_CONF)[0]
n_masks = len(result.masks) if result.masks is not None else 0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

ax1.imshow(np.array(Image.open(CUSTOM_IMAGE)))
ax1.set_title('Original', fontsize=12, fontweight='bold')
ax1.axis('off')

ax2.imshow(result.plot(boxes=False)[:, :, ::-1])
ax2.set_title(f'YOLOv26x — {n_masks} fields delineated', fontsize=12, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
if result.masks is not None and len(result.masks) > 0:
    print(f'\n🔎 {n_masks} fields detected:')
    for i, (mask, box) in enumerate(zip(result.masks.data, result.boxes)):
        conf = float(box.conf)
        area_px = int(mask.sum())
        print(f'   Field {i+1}: conf={conf:.2f}  area={area_px} px')
else:
    print('No fields detected. Try lowering INFER_CONF further.')